# Foundation B — Symbolic Lock Analysis

Test candidate Lagrangians for the mass-coupling lock.

For each model:
1. Define the Lagrangian symbolically
2. Perform canonical normalization
3. Compute physical mass and coupling
4. Test whether R = m/g_eff is constant (LOCKED) or parameter-dependent (UNLOCKED)

In [ ]:
import sympy as sp
from sympy import sqrt, symbols, simplify, diff, oo, limit, Rational

# Common symbols
M_Pl = symbols('M_Pl', positive=True)
t3 = symbols('t_3', positive=True)  # PGT torsion-squared coupling (|t_3|)
pi = sp.pi

## Lock Detection Function

Given m(params) and g_eff(params), compute R = m/g_eff and check if it depends on any parameter.

In [ ]:
def detect_lock(m_expr, g_expr, params, model_name="Model"):
    """Detect whether mass and coupling are locked.
    
    Args:
        m_expr: symbolic expression for physical mass
        g_expr: symbolic expression for physical coupling
        params: list of free parameters to check
        model_name: string label
    
    Returns:
        'LOCKED', 'PARTIALLY_UNLOCKED', or 'FULLY_UNLOCKED'
    """
    R = simplify(m_expr / g_expr)
    print(f"=== {model_name} ===")
    print(f"  m     = {m_expr}")
    print(f"  g_eff = {g_expr}")
    print(f"  R = m/g_eff = {R}")
    print()
    
    # Check if R depends on any parameter
    dependent_params = []
    for p in params:
        dR = simplify(diff(R, p))
        if dR != 0:
            dependent_params.append(p)
            print(f"  dR/d({p}) = {dR}  [R depends on {p}]")
        else:
            print(f"  dR/d({p}) = 0  [R independent of {p}]")
    
    print()
    if len(dependent_params) == 0:
        print(f"  VERDICT: LOCKED")
        print(f"  R is constant — mass and coupling cannot be independently adjusted.")
        return 'LOCKED'
    else:
        print(f"  VERDICT: UNLOCKED (R depends on {dependent_params})")
        print(f"  Mass and coupling CAN be independently adjusted.")
        return 'UNLOCKED'


## Baseline: PGT 0⁻ Axial Torsion Mode (Foundation A)

Known to be LOCKED. This validates our diagnostic.

In [ ]:
# PGT 0- mode
# L = -1/2 * |t_3| * (dB)^2 - 1/2 * (M_Pl^2 / (16*pi)) * B^2 + (1/M_Pl) * B * J
# Z = |t_3|, mu^2 = M_Pl^2/(16*pi), g_bare = 1/M_Pl

Z_pgt = t3
mu2_pgt = M_Pl**2 / (16 * pi)
g_bare_pgt = 1 / M_Pl

# Canonical normalization: B_can = sqrt(Z) * B
m_pgt = sqrt(mu2_pgt) / sqrt(Z_pgt)  # = M_Pl / (4*sqrt(pi*t3))
g_pgt = g_bare_pgt / sqrt(Z_pgt)     # = 1 / (M_Pl * sqrt(t3))

result_pgt = detect_lock(m_pgt, g_pgt, [t3, M_Pl], "PGT 0- mode")

## Model A: PGT + Higgs Portal

Add an independent scalar σ with VEV v that contributes to the torsion mass through a portal coupling λ σ² B².

In [ ]:
# Model A: PGT 0- mode + Higgs portal
# After σ gets VEV v:
# L = -1/2 * Z * (dB)^2 - 1/2 * (mu^2 + lambda*v^2) * B^2 + g_bare * B * J

lam, v = symbols('lambda v', positive=True)

Z_A = t3
mu2_A = M_Pl**2 / (16 * pi) + lam * v**2
g_bare_A = 1 / M_Pl

m_A = sqrt(mu2_A) / sqrt(Z_A)
g_A = g_bare_A / sqrt(Z_A)

result_A = detect_lock(m_A, g_A, [t3, lam, v, M_Pl], "Model A: PGT + Higgs portal")

In [ ]:
# Model A: Check the light-mass limit
# Can we have m -> 0 with g_eff finite?
# m = sqrt(M_Pl^2/(16*pi) + lambda*v^2) / sqrt(t3)
# g = 1/(M_Pl * sqrt(t3))
#
# m -> 0 requires: M_Pl^2/(16*pi) + lambda*v^2 -> 0 (impossible, both positive)
#                OR t3 -> infinity
# If t3 -> infinity: m -> 0 AND g -> 0. Still locked in the LIGHT-MASS LIMIT.
#
# But: if t3 ~ O(1) and lambda*v^2 dominates, then:
#   m ~ sqrt(lambda) * v / sqrt(t3)   (can be small if lambda*v^2 is small)
#   g ~ 1/(M_Pl * sqrt(t3))           (gravitational strength if t3 ~ O(1))
#
# The lock is broken in the sense that m depends on (lambda, v) while g depends on t3.
# But making m small requires small lambda*v^2 — a NEW hierarchy problem.

print("Model A: Light-mass analysis")
print("="*50)
print()

# Fix t3 = 1 (gravitational strength coupling)
m_A_t1 = m_A.subs(t3, 1)
g_A_t1 = g_A.subs(t3, 1)
print(f"At t3 = 1:")
print(f"  m = {simplify(m_A_t1)}")
print(f"  g = {simplify(g_A_t1)}")
print(f"  g ~ 1/M_Pl  (gravitational strength)")
print()
print(f"  m is small when lambda*v^2 << M_Pl^2/(16*pi)")
print(f"  m ~ sqrt(lambda)*v when portal term dominates")
print(f"  For m ~ meV ~ 10^-3 eV and M_Pl ~ 2.4*10^18 GeV:")
print(f"    sqrt(lambda)*v ~ meV = 10^-3 eV")
print(f"    This is achievable but requires a new small scale.")
print()
print("  VERDICT: PARTIALLY_UNLOCKED")
print("  Lock is broken (m depends on lambda,v; g depends on t3).")
print("  But the tiny mass is NOT natural — it requires small lambda*v^2.")

## Model B: ALP-like Pseudoscalar with Shift Symmetry

A pseudoscalar θ with:
- Kinetic term from geometric sector (decay constant f)
- Mass from shift-symmetry breaking (instanton scale Λ)
- Coupling to matter through geometric Nieh-Yan term (coupling α)

This mimics the QCD axion structure but with geometric origin.

In [ ]:
# Model B: Geometric ALP
# L = -1/2 * f^2 * (dtheta)^2 - Lambda^4 * (1 - cos(theta/f)) + alpha * theta * N4_matter
#
# Canonical normalization: theta_can = f * theta
# m_theta = Lambda^2 / f    (from expanding cosine potential)
# g_eff = alpha / f          (from Nieh-Yan coupling)
#
# Key: Lambda and alpha are INDEPENDENT parameters.

f, Lambda, alpha = symbols('f Lambda alpha', positive=True)

m_B = Lambda**2 / f
g_B = alpha / f

result_B = detect_lock(m_B, g_B, [f, Lambda, alpha], "Model B: Geometric ALP")

In [ ]:
# Model B: Detailed analysis
print("Model B: Detailed lock analysis")
print("="*50)
print()

R_B = simplify(m_B / g_B)
print(f"R = m/g = {R_B}")
print(f"R = Lambda^2 / alpha")
print()
print(f"R depends on Lambda and alpha, NOT on f.")
print(f"This means:")
print(f"  - Varying f changes BOTH m and g proportionally (locked direction)")
print(f"  - Varying Lambda changes m but NOT g (unlocked direction!)")
print(f"  - Varying alpha changes g but NOT m (unlocked direction!)")
print()
print(f"Light-mass limit: Lambda -> 0 with f, alpha fixed.")
print(f"  m -> 0 while g = alpha/f remains finite.")
print()
print(f"Mass protection: At Lambda = 0, the potential vanishes and the")
print(f"  shift symmetry theta -> theta + c is restored.")
print(f"  The mass is technically natural by 't Hooft's criterion.")
print()
print(f"VERDICT: FULLY_UNLOCKED")
print(f"  IF the geometric origin (Nieh-Yan coupling in MAG) produces")
print(f"  this structure with independent Lambda and alpha.")
print()
print(f"CRITICAL QUESTION: Does the Nieh-Yan form in metric-affine gravity")
print(f"  produce a coupling alpha that is genuinely independent of the")
print(f"  shift-symmetry-breaking scale Lambda? This is the mathematical")
print(f"  question that must be answered.")

## Model C: Two-Field PGT System

In [ ]:
# Model C: Two propagating torsion modes A (0-) and B (0+)
# with portal coupling lambda * A^2 * B^2
#
# In PGT, BOTH modes get their kinetic terms from the torsion-squared sector:
# Z_A = |t_3|,  Z_B = |t_2|
# mu_A^2 = M_Pl^2 / (16*pi*|t_3|) ... wait, this is the same structure
#
# After canonical normalization:
# m_A = M_Pl / sqrt(t_3)   (schematic)
# m_B = M_Pl / sqrt(t_2)   (schematic)
# g_A = 1/(M_Pl * sqrt(t_3))
#
# Portal contribution to A's mass from B's VEV:
# delta_m_A^2 = lambda * <B_can>^2 / Z_A
#
# But <B_can> itself comes from PGT and scales with PGT parameters.
# The portal coupling lambda is a quartic in the PGT action and is
# determined by the same t_1, t_2, t_3 couplings.

t2 = symbols('t_2', positive=True)
lam_portal = symbols('lambda_p', positive=True)

# Mode A mass with portal from mode B
# If B develops VEV: <B_can> ~ M_Pl / sqrt(t_2)  (at the PGT minimum)
# Portal mass: delta_m_A^2 = lam_portal * M_Pl^2 / (t_2 * t_3)

Z_C = t3
mu2_C = M_Pl**2 / (16*pi) + lam_portal * M_Pl**2 / (t2)
g_bare_C = 1/M_Pl

m_C = sqrt(mu2_C) / sqrt(Z_C)
g_C = g_bare_C / sqrt(Z_C)

result_C = detect_lock(m_C, g_C, [t3, t2, lam_portal, M_Pl], "Model C: Two-field PGT")

In [ ]:
# Model C analysis
print("Model C: Analysis")
print("="*50)
print()
print("R = m/g depends on t2 and lam_portal — parameters that g does NOT depend on.")
print("So the lock IS broken in principle.")
print()
print("But the problem is:")
print("  1. In PGT, lam_portal is NOT a free parameter — it is determined by t1, t2, t3.")
print("  2. Ghost-free two-mode PGT is severely constrained (Blagojevic-Cvetkovic).")
print("  3. The portal coupling and kinetic terms share the same PGT origin.")
print()
print("Even though R formally depends on t2, the PGT ghost-free conditions may")
print("impose relations between t2, t3, and lam_portal that re-lock the system.")
print()
print("VERDICT: FORMALLY UNLOCKED but likely re-locked by PGT constraints.")
print("Requires explicit ghost analysis of the two-mode PGT parameter space.")

## Summary of Lock Analysis Results

In [ ]:
print("\n" + "="*60)
print("LOCK ANALYSIS SUMMARY")
print("="*60)
print()
print(f"{'Model':<35} {'Lock Status':<25} {'Mass Natural?'}")
print("-"*75)
print(f"{'PGT 0- (baseline)':<35} {'LOCKED':<25} {'N/A'}")
print(f"{'A: PGT + Higgs portal':<35} {'PARTIALLY_UNLOCKED':<25} {'No (new hierarchy)'}")
print(f"{'B: Geometric ALP (shift sym)':<35} {'FULLY_UNLOCKED *':<25} {'Yes (shift symmetry)'}")
print(f"{'C: Two-field PGT':<35} {'FORMALLY_UNLOCKED **':<25} {'Unknown'}")
print()
print("* Model B is FULLY_UNLOCKED if the Nieh-Yan form in MAG is non-topological.")
print("  This is a mathematical question that has not been checked.")
print()
print("** Model C is formally unlocked but PGT ghost constraints may re-lock it.")
print("   Requires explicit multi-mode ghost analysis.")
print()
print("KEY FINDING: The ALP structure (Model B) is the only architecture that")
print("achieves FULL unlocking with natural mass. The question is whether")
print("any geometric theory produces this structure without collapsing to")
print("a generic (non-geometric) ALP.")